# Support Ticket Intelligence System

## Text Preprocessing

This notebook transforms raw customer support tickets into clean text data.

The preprocessing pipeline includes:

- Selecting relevant features
- Text normalization
- Tokenization
- Stopword removal
- Lemmatization
- POS tagging

The processed dataset will be used for:

- Ticket Type Classification
- Ticket Priority Classification
- Resolution Retrieval

In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

In [3]:
df = pd.read_csv(
    "../data/raw/customer_support_tickets.csv"
)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [4]:
selected_columns = [
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Ticket Channel",
    "Ticket Subject",
    "Ticket Description",
    "Resolution",
    "Ticket Type",
    "Ticket Priority"
]


df = df[selected_columns]

In [5]:
text_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Resolution"
]


df[text_columns] = (
    df[text_columns]
    .fillna("")
)

In [11]:
df['Ticket Description']

0       I'm having an issue with the {product_purchase...
1       I'm having an issue with the {product_purchase...
2       I'm facing a problem with my {product_purchase...
3       I'm having an issue with the {product_purchase...
4       I'm having an issue with the {product_purchase...
                              ...                        
8464    My {product_purchased} is making strange noise...
8465    I'm having an issue with the {product_purchase...
8466    I'm having an issue with the {product_purchase...
8467    I'm having an issue with the {product_purchase...
8468    There seems to be a hardware problem with my {...
Name: Ticket Description, Length: 8469, dtype: str

We need one main text column

In [13]:
df["combined_text"] = (
    df["Ticket Subject"]
    + " "
    + df["Ticket Description"]
)
df["combined_text"].head(5)

0    Product setup I'm having an issue with the {pr...
1    Peripheral compatibility I'm having an issue w...
2    Network problem I'm facing a problem with my {...
3    Account access I'm having an issue with the {p...
4    Data loss I'm having an issue with the {produc...
Name: combined_text, dtype: str

### Text Cleaning Function

- Operations:

    - lowercase
    - remove symbols
    - remove extra spaces

In [14]:
def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z\s]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [15]:
df["clean_text"] = (
    df["combined_text"]
    .apply(clean_text)
)

### Tokenization

In [16]:
df["tokens"] = (
    df["clean_text"]
    .apply(
        nltk.word_tokenize
    )
)

### Stopword Removal

In [17]:
stop_words = set(
    stopwords.words("english")
)


df["filtered_tokens"] = (
    df["tokens"]
    .apply(
        lambda tokens:
        [
            word
            for word in tokens
            if word not in stop_words
        ]
    )
)

### Lemmatization

In [18]:
lemmatizer = WordNetLemmatizer()


df["lemmatized_tokens"] = (
    df["filtered_tokens"]
    .apply(
        lambda tokens:
        [
            lemmatizer.lemmatize(word)
            for word in tokens
        ]
    )
)

In [19]:
df["processed_text"] = (
    df["lemmatized_tokens"]
    .apply(
        lambda x:
        " ".join(x)
    )
)

### POS Tagging

In [21]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")

df["pos_tags"] = (
    df["lemmatized_tokens"]
    .apply(pos_tag)
)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/bharath/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


### Remove Temporary Columns

In [22]:
df.drop(
    columns=[
        "tokens",
        "filtered_tokens",
        "lemmatized_tokens"
    ],
    inplace=True
)

### Save Processed Dataset

In [23]:
df.to_csv(
    "../data/processed/clean_tickets.csv",
    index=False
)